# 사고 빈도 모델링 1~6단계 (포아송 vs 음이항 GLM)

> 사고빈도_모델링_1_4단계.ipynb의 복사본. 0~4단계는 원본과 같고, 5~6단계에서 음이항 GLM으로 과대산포를 보완해 비교함

| 단계 | 하는 일 | 전처리는 어디서? |
|---|---|---|
| 0 | 데이터 불러오기 + **고정 규칙** 전처리 (cap, Area 제거, 나이 구간, log) | 데이터를 보고 배우는 게 없음 → 분할 **전**에 해도 누수 없음 |
| 1 | 스케일링할 피처 확인 + 전처리 **설계** (Other 합치기, 표준화, 원-핫) | 여기서는 **정의만** 하고 fit하지 않음 |
| 2 | X, Y를 새 DataFrame으로 추출 | |
| 3 | train_test_split으로 분리 (같은 계약은 같은 쪽으로) | |
| 4 | fit() · predict() (statsmodels Poisson 회귀, offset = log 가입 기간) | 1단계에서 설계한 전처리가 **train으로만** fit됨 |
| 5 | 교차 검증 후 성능 비교 (포아송 vs 음이항, GroupKFold 5겹) | 겹마다 전처리를 다시 배움 |
| 6 | GridSearchCV로 음이항의 분산 파라미터 alpha 탐색 → test 최종 평가 | |

statsmodels 모델을 sklearn 모양(`fit` · `predict`)으로 감싸서 `Pipeline`에 넣음
→ 5단계 교차 검증·6단계 GridSearchCV를 sklearn 그대로 쓰면서, 겹마다 전처리를 자동으로 다시 배움 (누수 없음)

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_poisson_deviance, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# ================================================================ 0단계: 불러오기 + 고정 규칙 전처리
# 고정된 숫자(4, 1, 20, 150, 12, 나이 경계)를 행마다 적용할 뿐 → 분할 전에 해도 누수 없음
freq = pd.read_csv("freMTPL2freq.csv")
print(f"계약 수: {len(freq):,} · 사고 {freq.ClaimNb.sum():,}건 · "
      f"1년당 사고율 {freq.ClaimNb.sum() / freq.Exposure.sum():.4f}")

# ---------------------------------------------------------------- 묶음 번호 (3단계 분할용)
# 한 계약이 여러 IDpol로 쪼개져 있음 → 특성이 모두 같은 행 = 같은 묶음
# 자르기(clip) 전 원본 값으로 만들어야 서로 다른 계약이 덜 섞임
FEAT = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand",
        "VehGas", "Density", "Region"]
freq["GroupID"] = freq.groupby(FEAT, sort=False).ngroup()
#  .ngroup으로 번호 부여

# ---------------------------------------------------------------- 타깃 · 가입 기간
# Y = ClaimNb(사고 건수), 가입 기간은 4단계에서 offset = log(Exposure)로 씀
# 사고 5건 이상 9건은 기록 오류로 보임 → 지우지 않고 4로 cap ("위험하다"는 정보는 남김)
freq["ClaimNb"] = freq["ClaimNb"].clip(upper=4)
# 1년 넘는 계약은 기록 오차 → 1로 cap (분모가 커지면 사고율이 작게 계산됨)
freq["Exposure"] = freq["Exposure"].clip(upper=1)

# ---------------------------------------------------------------- 피처
# Area는 Density를 6구간으로 자른 것 → 같은 정보가 두 번 들어가므로 제거
freq = freq.drop(columns=["Area"])
AGE_BINS = [17, 21, 25, 30, 40, 50, 60, 70, 80, 101]
freq["VehAge"] = freq["VehAge"].clip(upper=20)
freq["BonusMalus"] = freq["BonusMalus"].clip(upper=150)
freq["DrivAgeBin"] = pd.cut(freq["DrivAge"], bins=AGE_BINS).astype(str)
freq["VehPower"] = freq["VehPower"].clip(upper=12).astype(str)   # 등급별로 따로 보려고 범주로
freq["logDensity"] = np.log(freq["Density"])

# Exposure = 0이면 offset log(0) = -inf → numpy는 경고만 내고 계속 계산함
assert freq["Exposure"].between(0, 1, inclusive="right").all(), "Exposure가 0 이하 또는 1 초과"
assert "nan" not in set(freq["DrivAgeBin"]), "나이 구간에 안 들어간 운전자 있음"
print("[0] 고정 규칙 전처리 완료")

계약 수: 678,013 · 사고 36,102건 · 1년당 사고율 0.1007
[0] 고정 규칙 전처리 완료


In [3]:
# ================================================================ 1단계: 스케일링할 피처 확인
# 숫자 피처의 범위·치우침(skew)을 보고 정규화(MinMax)와 표준화(Standard) 중 고름
check = ["VehAge", "BonusMalus", "Density", "logDensity", "Exposure"]
desc = freq[check].describe().T[["min", "max", "mean", "std"]]
desc["skew"] = freq[check].skew()
print("[1] 숫자 피처 확인")
print(desc.round(2).to_string())

[1] 숫자 피처 확인
             min      max     mean      std  skew
VehAge       0.0     20.0     6.98     5.40  0.52
BonusMalus  50.0    150.0    59.76    15.61  1.70
Density      1.0  27000.0  1792.42  3958.65  4.65
logDensity   0.0     10.2     5.98     1.87  0.03
Exposure     0.0      1.0     0.53     0.36  0.08


**확인 결과 → 결정**

| 피처 | 처리 | 이유 |
|---|---|---|
| VehAge, BonusMalus, logDensity | **표준화 (StandardScaler)** | 단위가 제각각(년·점·log 인구)이라 크기를 맞춤 → 계수를 "1 표준편차당 사고율 배수"로 같은 잣대에서 비교. 6단계 규제(alpha)가 모든 계수에 공평하게 걸리려면 필수. BonusMalus처럼 한쪽으로 몰린 값은 MinMax로 하면 대부분이 0 근처로 눌림 |
| Density | log 변환 후 표준화 (원본은 안 씀) | 1~27,000으로 꼬리가 아주 길어서(skew 큼) 그대로 표준화하면 소수 대도시가 계수를 좌우함 |
| Exposure | 스케일링 안 함 | 피처가 아니라 **offset**(log 가입 기간, 계수 1로 고정). 단위가 '년'이어야 의미가 있음 |
| ClaimNb (Y) | 스케일링 안 함 | 타깃. Poisson 회귀는 0, 1, 2…로 세는 값을 그대로 받아야 함 |
| 범주형 (나이 구간, VehPower, VehGas, VehBrand, Region) | 원-핫 인코딩만 | 0/1 값이라 스케일링 의미 없음 |

In [4]:
# ---------------------------------------------------------------- 1단계: 데이터에서 "배우는" 전처리 + 모델 설계
# 아래는 정의만 함 (fit 안 함) → 4단계 fit()에서 train으로만 배움
NUM_COLS = ["VehAge", "BonusMalus", "logDensity"]
# 연속형 피쳐
CAT_COLS = ["DrivAgeBin", "VehPower", "VehGas", "VehBrand", "Region"]
# 범주형 피쳐


class RareLevelMerger(BaseEstimator, TransformerMixin):
    """사고 수가 min_claims 미만인 범주를 Other로 합침
    fit: 어떤 범주를 남길지 배움 (train에서만) / transform: 배운 목록으로 바꿈 (어디에나)"""
    #BaseEstimator는 get_params와 set_params(GridSearchCV가 쓰는 기능)를 줌.
    #TransformerMixin은 fit_transform(fit과 transform을 한 번에)을 줌.

    def __init__(self, cols=("Region", "VehBrand"), min_claims=200):
        self.cols = cols
        self.min_claims = min_claims
        # 기본 설정값

    def fit(self, X, y):
        # 희귀 범주는 행 수가 아니라 "사고 수(Y)"로 판단
        # (사고 45건인 지역은 5건만 더 나도 사고율이 11% 흔들림)
        claims = pd.Series(np.asarray(y), index=X.index)
        self.keep_ = {}
        # 200이상인 지역이나 브랜드 남기기
        for c in self.cols:
            total = claims.groupby(X[c]).sum()
            # 같은 지역마다 group하고 사고 건수 합하기
            self.keep_[c] = set(total[total >= self.min_claims].index)
            # 200 이상인 지역이나 브랜드 이름 남기기
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            X[c] = X[c].where(X[c].isin(self.keep_[c]), "Other")
            # 남은 지역이나 브랜드(self.keep_[c])에 없는 이름이라면 Other이라 적기
        return X


class SMPoissonGLM(BaseEstimator, RegressorMixin):
    """statsmodels Poisson 회귀를 sklearn 모양(fit · predict)으로 감싼 것
    X의 Exposure 열 → offset = log(Exposure), 나머지 열 → 피처
    alpha = 0이면 일반 Poisson 회귀(summary로 p값 확인 가능), alpha > 0이면 L2(릿지) 규제"""

    def __init__(self, alpha=0.0):
        self.alpha = alpha
        # 기본 설정 람다 값(릿지회귀 대비용)

    def _split(self, X):
        X = pd.DataFrame(X).copy()
        exposure = X.pop("Exposure") if "Exposure" in X else X.pop(X.columns[-1])
        return sm.add_constant(X, has_constant="add"), np.log(np.asarray(exposure, dtype=float))
        # constant = 1인 걸로 다 채운 X 반환 / exposure에 log 취한 값 반환

    def fit(self, X, y):
        exog, offset = self._split(X)
        # 위에 했던 나누기 나눠야 하는 이유는 아래 모형 함수는 exposure까지 포함해서 결과를 내기 때문
        # offset를 따로 둬야하기 때문
        model = sm.GLM(np.asarray(y, dtype=float), exog, family=sm.families.Poisson(), offset=offset)
        if self.alpha == 0:
            self.result_ = model.fit()
        else:
            penalty = np.r_[0.0, np.full(exog.shape[1] - 1, self.alpha)]   # 절편은 규제하지 않음
            self.result_ = model.fit_regularized(alpha=penalty, L1_wt=0)   # L1_wt=0 → L2(릿지)
        self.params_ = np.asarray(self.result_.params)
        return self

    def predict(self, X):
        # 예상 사고 건수 = exp(계수 합 + log 가입 기간) = 가입 기간 × 1년당 사고율
        exog, offset = self._split(X)
        return np.exp(exog.to_numpy(dtype=float) @ self.params_ + offset)
        # 각 계수 반환


def make_model(estimator):
    # 희귀 범주 합치기 → 숫자는 표준화 · 범주는 원-핫 · Exposure는 그대로 통과 → 모델
    encode = ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        # 표준화
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CAT_COLS),
        # 범주 변수화
        ("offset", "passthrough", ["Exposure"]),      # 스케일링 안 하고 모델에 넘김 (offset용)
    ], verbose_feature_names_out=False).set_output(transform="pandas")   # 열 이름 유지 → summary에 표시
    return Pipeline([("rare", RareLevelMerger()), ("encode", encode), ("model", estimator)])


print("[1] 스케일링 대상:", NUM_COLS)
print("    원-핫 대상:", CAT_COLS)
print("    offset: log(Exposure)")

[1] 스케일링 대상: ['VehAge', 'BonusMalus', 'logDensity']
    원-핫 대상: ['DrivAgeBin', 'VehPower', 'VehGas', 'VehBrand', 'Region']
    offset: log(Exposure)


In [5]:
# ================================================================ 2단계: X, Y를 새 DataFrame으로 추출
# X = 피처 8개 + Exposure(offset용), Y = 사고 건수
X = freq[["Exposure"] + NUM_COLS + CAT_COLS].copy()
y = freq["ClaimNb"].copy()
groups = freq["GroupID"].copy()
print(f"[2] X {X.shape[0]:,}행 × {X.shape[1]}열 · Y = ClaimNb")
print(X.head(3).to_string())

[2] X 678,013행 × 9열 · Y = ClaimNb
   Exposure  VehAge  BonusMalus  logDensity DrivAgeBin VehPower   VehGas VehBrand       Region
0      0.10       0          50    7.104144   (50, 60]        5  Regular      B12  Rhone-Alpes
1      0.77       0          50    7.104144   (50, 60]        5  Regular      B12  Rhone-Alpes
2      0.75       2          50    3.988984   (50, 60]        6   Diesel      B12     Picardie


In [6]:
# ================================================================ 3단계: train_test_split으로 분리
# 행이 아니라 "묶음 번호"를 train_test_split으로 나눔 → 같은 계약이 train과 test에 동시에 들어가지 않음
# 묶음에 사고가 있었는지(0/1)로 층화 → 사고 난 계약이 양쪽에 고르게
group_claim = freq.groupby("GroupID")["ClaimNb"].max().clip(upper=1)
g_train, g_test = train_test_split(group_claim.index, test_size=0.2, random_state=42,
                                   stratify=group_claim)
is_train = groups.isin(g_train)

X_train, X_test = X[is_train].copy(), X[~is_train].copy()
y_train, y_test = y[is_train].copy(), y[~is_train].copy()
groups_train = groups[is_train]          # 5단계 교차 검증도 묶음 단위로 (GroupKFold)

assert set(groups[is_train]).isdisjoint(groups[~is_train]), "같은 묶음이 train과 test에 모두 있음"
print(f"[3] train {len(X_train):,} / test {len(X_test):,} (test 비율 {len(X_test) / len(X):.1%})")
print(f"    1년당 사고율 train {y_train.sum() / X_train.Exposure.sum():.4f}"
      f" / test {y_test.sum() / X_test.Exposure.sum():.4f}")

[3] train 542,576 / test 135,437 (test 비율 20.0%)
    1년당 사고율 train 0.1005 / test 0.1009


In [7]:
# ================================================================ 4단계: fit() · predict() — statsmodels Poisson 회귀
# log(예상 사고 수) = log(가입 기간) + β0 + β1·VehAge + …  → 예측은 항상 0보다 큼
# fit() 한 번에: 희귀 범주 목록·평균·표준편차를 train으로 배우고 → 모델 학습
# alpha = 0: 규제 없는 일반 Poisson 회귀 (6단계에서는 음이항 GLM의 분산 파라미터 nb_alpha를 탐색)
pois = make_model(SMPoissonGLM(alpha=0))
pois.fit(X_train, y_train)

pred_train = pois.predict(X_train)      # 예상 사고 건수
pred_test = pois.predict(X_test)


def evaluate(y_true, pred):
    # Poisson deviance = Poisson 회귀가 최소화하는 기준 → 이 모델의 주 지표 (작을수록 좋음)
    return {"Poisson deviance": mean_poisson_deviance(y_true, pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
            "R²": r2_score(y_true, pred)}


result = pd.DataFrame({"train": evaluate(y_train, pred_train),
                       "test": evaluate(y_test, pred_test)}).T
print("[4] Poisson 회귀 결과")
print(result.round(4).to_string())
print(f"    예상 사고 건수 합: test {pred_test.sum():,.0f}건 / 실제 {y_test.sum():,}건")

[4] Poisson 회귀 결과
       Poisson deviance    RMSE      R²
train            0.3181  0.2364  0.0111
test             0.3209  0.2379  0.0093
    예상 사고 건수 합: test 7,222건 / 실제 7,246건


In [8]:
# ---------------------------------------------------------------- 4단계: 계수 확인 (statsmodels의 장점)
# summary: 계수 · 표준오차 · p값 · 95% 신뢰구간
res = pois.named_steps["model"].result_
print(res.summary())

# 표준화한 피처: exp(계수) = "1 표준편차 늘 때 사고율 몇 배"
ci = np.exp(res.conf_int().loc[NUM_COLS])
print("\n[4] 1 표준편차 늘 때 사고율 배수 (95% 신뢰구간)")
print(pd.DataFrame({"배수": np.exp(res.params[NUM_COLS]), "하한": ci[0], "상한": ci[1]}).round(3).to_string())

# 7단계 PPT용: train에서 배운 전처리 기준
rare = pois.named_steps["rare"]
scaler = pois.named_steps["encode"].named_transformers_["num"]
print("\n    Other로 합친 Region:", sorted(set(X_train.Region) - rare.keep_["Region"]))
print("    Other로 합친 VehBrand:", sorted(set(X_train.VehBrand) - rare.keep_["VehBrand"]))
print(pd.DataFrame({"평균": scaler.mean_, "표준편차": scaler.scale_}, index=NUM_COLS).round(3).to_string())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:               542576
Model:                            GLM   Df Residuals:                   542528
Model Family:                 Poisson   Df Model:                           47
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1400e+05
Date:                Tue, 27 Oct 2026   Deviance:                   1.7260e+05
Time:                        18:39:59   Pearson chi2:                 1.38e+06
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01156
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

In [9]:
# ---------------------------------------------------------------- 참고: 변수 없는 모델과 비교
# 모두에게 train의 1년당 사고율을 주는 모델 (예상 사고 수 = 평균 사고율 × 가입 기간)
# → Poisson 회귀가 이보다 나아야 피처가 쓸모 있음
# (R²가 작게 나와도 정상: 사고는 우연이 커서 개인 단위로는 원래 잘 안 맞음)
base_rate = y_train.sum() / X_train["Exposure"].sum()
pred_base = base_rate * X_test["Exposure"]
compare = pd.DataFrame({"변수 없음": evaluate(y_test, pred_base),
                        "Poisson 회귀": evaluate(y_test, pred_test)}).T
print("[참고] test 비교")
print(compare.round(4).to_string())
gain = 1 - compare.loc["Poisson 회귀", "Poisson deviance"] / compare.loc["변수 없음", "Poisson deviance"]
print(f"    Poisson deviance가 변수 없는 모델보다 {gain:.1%} 작음")

[참고] test 비교
            Poisson deviance    RMSE      R²
변수 없음                 0.3324  0.2393 -0.0028
Poisson 회귀            0.3209  0.2379  0.0093
    Poisson deviance가 변수 없는 모델보다 3.5% 작음


## 5~6단계: 포아송 vs 음이항 GLM (과대산포 보완)

**과대산포란?**
- 포아송은 "분산 = 평균"을 가정함. 실제 분산이 평균보다 크면 과대산포
- 과대산포가 있으면 예측값(평균)은 크게 안 틀리지만, 계수의 표준오차·p값이 실제보다 작게(과하게 확신) 나옴

**음이항(Negative Binomial) GLM**
- 분산 = 평균 + alpha × 평균² → **alpha가 과대산포 크기를 조절하는 파라미터** (alpha → 0이면 포아송과 같음)
- 사람마다 위험도가 달라서 사고가 일부에게 몰리는 "건수 데이터"의 과대산포에 쓰는 정석 모델
- 4단계와 같은 statsmodels GLM이라 offset(log 가입 기간)을 그대로 쓰고, summary로 p값도 볼 수 있음

| 단계 | 셀 | 하는 일 |
|---|---|---|
| 5·6 준비 | 아래 첫 번째 셀 | 음이항 GLM을 sklearn 모양으로 감싸기, 평가 기준(Poisson deviance·RMSE·R²·로그우도), GroupKFold |
| 5 준비 | 두 번째 셀 | 과대산포가 어디서 생기는지 확인 + 최대우도로 alpha 추정 |
| 5 | 세 번째 셀 | 포아송 vs 음이항을 GroupKFold 5겹 교차 검증으로 성능 비교 |
| 6 | 네 번째 셀 | GridSearchCV로 음이항의 alpha 탐색 |
| 6 마무리 | 다섯·여섯 번째 셀 | test로 딱 한 번 최종 평가 + p값(유의한 계수 수) 비교 |

In [10]:
# ================================================================ 5·6단계 준비: 음이항 GLM과 평가 기준
from scipy import stats
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_validate


class SMNegBinGLM(SMPoissonGLM):
    """[5·6단계 준비] statsmodels 음이항 GLM을 sklearn 모양(fit · predict)으로 감싼 것
    1단계 SMPoissonGLM을 상속 → X에서 Exposure를 꺼내 offset으로 쓰는 방법(_split)과 predict는 그대로 사용
    nb_alpha: 분산 = 평균 + nb_alpha × 평균² 의 nb_alpha (6단계 GridSearchCV로 탐색할 파라미터)"""

    def __init__(self, nb_alpha=1.0):
        self.nb_alpha = nb_alpha

    def fit(self, X, y):
        exog, offset = self._split(X)
        family = sm.families.NegativeBinomial(alpha=self.nb_alpha)   # 링크는 포아송과 같은 log
        self.result_ = sm.GLM(np.asarray(y, dtype=float), exog, family=family, offset=offset).fit()
        self.params_ = np.asarray(self.result_.params)
        return self


# ---------------------------------------------------------------- 5·6단계 준비: 평가 기준
# 예측값 비교: 4단계와 같은 Poisson deviance · RMSE · R² (sklearn 기본 제공 이름 사용)
# 분산 비교: 로그우도(log-likelihood) = "실제로 나온 사고 수가 모델이 보기에 얼마나 그럴듯한가" (클수록 좋음)
#   → Poisson deviance는 평균(예측값)만 보고 분산은 안 봐서 alpha를 고르는 기준이 되지 못함
#   → 분산까지 반영하는 로그우도가 alpha를 고르는 올바른 기준
def mean_loglik(est, X, y):
    mu = np.asarray(est.predict(X), dtype=float)                # 예상 사고 건수
    a = getattr(est.named_steps["model"], "nb_alpha", 0.0)      # 포아송(SMPoissonGLM)은 0
    if a == 0:
        return stats.poisson.logpmf(y, mu).mean()
    n = 1 / a                                                   # 음이항: 분산 = mu + a·mu²
    return stats.nbinom.logpmf(y, n, n / (n + mu)).mean()


SCORING = {"poisson_dev": "neg_mean_poisson_deviance",          # sklearn은 클수록 좋게 → 음수로 반환
           "rmse": "neg_root_mean_squared_error",
           "r2": "r2",
           "loglik": mean_loglik}

# [5·6단계 준비] 동시에 학습할 개수. statsmodels GLM은 학습 한 번에 메모리를 많이 써서
#   -1(코어 수만큼)로 하면 메모리가 부족해질 수 있음 → 2개로 제한 (Colab 무료도 코어 2개)
N_JOBS = 2

# [5·6단계 준비] 교차 검증도 3단계처럼 "묶음 단위": 같은 계약이 학습 겹과 검증 겹에 동시에 들어가지 않게
cv = GroupKFold(n_splits=5)
print("[5·6 준비] 음이항 GLM 정의 · 평가 기준 4개 · GroupKFold 5겹")

[5·6 준비] 음이항 GLM 정의 · 평가 기준 4개 · GroupKFold 5겹


In [11]:
# ================================================================ 5단계 준비: 과대산포 확인 + alpha 추정
# [5단계 준비] 산포 지수 = Pearson chi2 / 자유도. 1이면 포아송 가정이 맞고, 1보다 크면 과대산포
res_p = pois.named_steps["model"].result_                      # 4단계 포아송 결과
print(f"[5 준비] 포아송 산포 지수 {res_p.pearson_chi2 / res_p.df_resid:.2f}")

# [5단계 준비] 과대산포가 어디서 생기는지: 가입 기간 구간별 Pearson 잔차² 평균 (1 근처면 포아송 가정이 맞음)
mu_tr = pois.predict(X_train)
pearson_sq = (y_train - mu_tr) ** 2 / mu_tr
exp_bin = pd.cut(X_train["Exposure"], [0, 0.1, 0.25, 0.5, 0.75, 0.99, 1.0])
by_exp = pearson_sq.groupby(exp_bin, observed=True).agg(["size", "mean", "sum"])
by_exp["chi2 비중"] = by_exp["sum"] / by_exp["sum"].sum()
print("    가입 기간 구간별 Pearson 잔차² 평균과 chi2 비중")
print(by_exp[["size", "mean", "chi2 비중"]].round(3).to_string())

# [5단계 준비] 최대우도(MLE)로 alpha 추정: 데이터가 가장 그럴듯해지는 alpha를 직접 계산
# GLM은 alpha를 고정값으로 받기 때문에, alpha까지 함께 추정하는 sm.NegativeBinomial(NB2)을 한 번 사용
prep = Pipeline(pois.steps[:2])                                 # 4단계에서 train으로 배운 전처리 재사용
Z = prep.transform(X_train)
offset_tr = np.log(Z.pop("Exposure").to_numpy())
exog_tr = sm.add_constant(Z, has_constant="add")
nb_mle = sm.NegativeBinomial(y_train.to_numpy(), exog_tr, offset=offset_tr, loglike_method="nb2").fit(
    start_params=np.r_[res_p.params.to_numpy(), 1.0],           # 포아송 계수에서 출발 → 수렴이 빠르고 안정적
    method="newton", maxiter=100, disp=0)
alpha_mle = float(np.asarray(nb_mle.params)[-1])
print(f"\n    최대우도로 추정한 alpha = {alpha_mle:.3f} (수렴: {nb_mle.mle_retvals['converged']})")

[5 준비] 포아송 산포 지수 2.55
    가입 기간 구간별 Pearson 잔차² 평균과 chi2 비중
                size   mean  chi2 비중
Exposure                            
(0.0, 0.1]    102404  8.401    0.622
(0.1, 0.25]    76222  2.130    0.117
(0.25, 0.5]   105023  1.270    0.096
(0.5, 0.75]    73954  1.024    0.055
(0.75, 0.99]   49753  0.904    0.032
(0.99, 1.0]   135220  0.791    0.077

    최대우도로 추정한 alpha = 0.839 (수렴: True)


In [12]:
# ================================================================ 5단계: 교차 검증 후 성능 비교 (포아송 vs 음이항)
# [5단계] 같은 겹(GroupKFold 5겹)에서 두 모델을 평가해 평균 ± 겹별 흔들림으로 비교
candidates = {"포아송": make_model(SMPoissonGLM(alpha=0)),
              f"음이항 (alpha={alpha_mle:.2f}, MLE)": make_model(SMNegBinGLM(nb_alpha=alpha_mle))}
rows = {}
for name, model in candidates.items():
    out = cross_validate(model, X_train, y_train, groups=groups_train, cv=cv, scoring=SCORING, n_jobs=N_JOBS)
    rows[name] = {
        "Poisson deviance": f"{-out['test_poisson_dev'].mean():.4f} ± {out['test_poisson_dev'].std():.4f}",
        "RMSE": f"{-out['test_rmse'].mean():.4f} ± {out['test_rmse'].std():.4f}",
        "R²": f"{out['test_r2'].mean():.4f} ± {out['test_r2'].std():.4f}",
        "로그우도(평균)": f"{out['test_loglik'].mean():.5f} ± {out['test_loglik'].std():.5f}",
    }
print("[5] GroupKFold 5겹 교차 검증 (deviance·RMSE 작을수록, R²·로그우도 클수록 좋음)")
print(pd.DataFrame(rows).T.to_string())

[5] GroupKFold 5겹 교차 검증 (deviance·RMSE 작을수록, R²·로그우도 클수록 좋음)
                      Poisson deviance             RMSE               R²            로그우도(평균)
포아송                    0.3184 ± 0.0038  0.2364 ± 0.0014  0.0108 ± 0.0019  -0.21023 ± 0.00245
음이항 (alpha=0.84, MLE)  0.3184 ± 0.0038  0.2365 ± 0.0014  0.0102 ± 0.0020  -0.20977 ± 0.00244


In [13]:
# ================================================================ 6단계: GridSearchCV로 음이항의 alpha 탐색
# [6단계] alpha: 분산 = 평균 + alpha × 평균². 클수록 과대산포를 크게 반영
# [6단계] 선택 기준(refit)은 로그우도: 분산까지 보는 기준이라 alpha를 고르는 데 맞음
param_grid = {"model__nb_alpha": [0.1, 0.25, 0.5, round(alpha_mle, 2), 1.0, 2.0, 5.0]}
grid = GridSearchCV(make_model(SMNegBinGLM()), param_grid, cv=cv, scoring=SCORING,
                    refit="loglik", n_jobs=N_JOBS)
# [6단계] groups → GroupKFold로 전달 (같은 계약이 학습 겹과 검증 겹에 섞이지 않게)
grid.fit(X_train, y_train, groups=groups_train)

res = pd.DataFrame(grid.cv_results_).set_index("param_model__nb_alpha")
table = pd.DataFrame({"로그우도(클수록 좋음)": res["mean_test_loglik"],
                      "Poisson deviance": -res["mean_test_poisson_dev"],
                      "RMSE": -res["mean_test_rmse"],
                      "R²": res["mean_test_r2"]})
print("[6] alpha별 교차 검증 평균")
print(table.round(5).to_string())
print(f"\n    최적 alpha: {grid.best_params_['model__nb_alpha']} (로그우도 {grid.best_score_:.5f})")
# [6단계] 해석: 로그우도는 alpha에 따라 달라지지만, deviance·RMSE·R²는 거의 같음
#         → alpha는 예측값(평균)이 아니라 "얼마나 흩어지는지(분산)"를 바꾸는 파라미터이기 때문

[6] alpha별 교차 검증 평균
                       로그우도(클수록 좋음)  Poisson deviance     RMSE       R²
param_model__nb_alpha                                                  
0.10                       -0.21010           0.31836  0.23641  0.01069
0.25                       -0.20997           0.31836  0.23643  0.01059
0.50                       -0.20983           0.31837  0.23645  0.01041
0.84                       -0.20977           0.31838  0.23648  0.01017
1.00                       -0.20978           0.31838  0.23649  0.01005
2.00                       -0.21014           0.31843  0.23658  0.00933
5.00                       -0.21234           0.31868  0.23684  0.00711

    최적 alpha: 0.84 (로그우도 -0.20977)


In [14]:
# ================================================================ 6단계 마무리: test 최종 평가 (딱 한 번)
# [6단계 마무리] GridSearchCV가 최적 alpha로 train 전체를 다시 학습해 둔 모델(best_estimator_) 사용
nb_best = grid.best_estimator_
pred_test_nb = nb_best.predict(X_test)                          # 예상 사고 건수

final = pd.DataFrame({
    "4단계 포아송": {**evaluate(y_test, pred_test), "로그우도": mean_loglik(pois, X_test, y_test)},
    f"6단계 음이항 (alpha={grid.best_params_['model__nb_alpha']})":
        {**evaluate(y_test, pred_test_nb), "로그우도": mean_loglik(nb_best, X_test, y_test)},
}).T
print("[6 마무리] test 비교")
print(final.round(5).to_string())
print(f"    예상 사고 건수 합: 포아송 {pred_test.sum():,.0f}건 / 음이항 {pred_test_nb.sum():,.0f}건 / 실제 {y_test.sum():,}건")

[6 마무리] test 비교
                      Poisson deviance     RMSE       R²     로그우도
4단계 포아송                        0.32087  0.23789  0.00928 -0.21184
6단계 음이항 (alpha=0.84)           0.32087  0.23796  0.00866 -0.21131
    예상 사고 건수 합: 포아송 7,222건 / 음이항 7,302건 / 실제 7,246건


In [15]:
# ---------------------------------------------------------------- 6단계 마무리: p값 비교 (과대산포 보정 효과)
# [6단계 마무리] 과대산포를 무시하면 표준오차가 작게 나와서 "유의하다(p < 0.05)"가 과하게 나옴
# 세 가지를 비교: 4단계 포아송 / 포아송 + scale="X2"(산포 지수만큼 표준오차를 키우는 보정) / 음이항
res_x2 = sm.GLM(y_train.to_numpy(), exog_tr, family=sm.families.Poisson(), offset=offset_tr).fit(scale="X2")
res_nb = nb_best.named_steps["model"].result_
fits = {"포아송": res_p, "포아송 + scale='X2'": res_x2, "음이항": res_nb}

sig = pd.DataFrame({name: {"유의한 계수 수 (p < 0.05)": int((r.pvalues.drop("const") < 0.05).sum()),
                           "전체 계수 수": len(r.params) - 1,
                           "표준오차 (포아송 대비 중앙값)": float(np.median(np.asarray(r.bse) / np.asarray(res_p.bse)))}
                    for name, r in fits.items()}).T
print("[6 마무리] 계수 유의성 비교 (train)")
print(sig.round(3).to_string())

# [6단계 마무리] 표준화한 숫자 피처: exp(계수) = "1 표준편차 늘 때 사고율 몇 배" (95% 신뢰구간)
ci_nb = np.exp(res_nb.conf_int().loc[NUM_COLS])
print("\n    음이항: 1 표준편차 늘 때 사고율 배수")
print(pd.DataFrame({"배수": np.exp(res_nb.params[NUM_COLS]), "하한": ci_nb[0], "상한": ci_nb[1]}).round(3).to_string())

[6 마무리] 계수 유의성 비교 (train)
                  유의한 계수 수 (p < 0.05)  전체 계수 수  표준오차 (포아송 대비 중앙값)
포아송                              23.0     47.0              1.000
포아송 + scale='X2'                 16.0     47.0              1.597
음이항                              22.0     47.0              1.031

    음이항: 1 표준편차 늘 때 사고율 배수
               배수     하한     상한
VehAge      0.801  0.790  0.813
BonusMalus  1.444  1.427  1.462
logDensity  1.085  1.070  1.101
